<a href="https://colab.research.google.com/github/Prab999/metacritic-vs-emmy-tvshow-analysis/blob/main/Datasetclean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data ETL & Cleaning Pipeline
This file is used to Standardize and merge two distinct Kaggle datasets(Metacritic TV Shows and Emmy Awards) to create a reliable relational database schema.

Due to the lack of a shared primary key between the two datasets, this notebook conducts a string-matching pipeline to normalize show titles (handling accents, punctuation, and spacing). The finalized data is then exported into Third Normal Form (3NF) compliant CSVs (`shows.csv` and `awards.csv`) to be loaded into a PostgreSQL database.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
#read in datassets
data = pd.read_csv('metacritic_tv_shows.csv', encoding='latin-1')
data.head()

,id,title,releaseDate,seasonCount,rating,genres,description,duration,tagline,metascore,metascore_count,metascore_sentiment,userscore,userscore_count,userscore_sentiment,created_by,production_companies,director,writer,top_cast
0,1000358361,Planet Earth: Blue Planet II,2017-10-29,1.0,TV-G,Documentary,"Airing simultaneously on AMC, BBC America, IFC...",50.0,Take a deep breath,97.0,7,Universal acclaim,82,178,Universal acclaim,NaN,"BBC Natural History Unit (NHU),BBC Studios,BBC...",James Honeyborne,NaN,"David Attenborough,Peter Drost,Roger Munns,Rog..."
1,1000359012,America to Me,2018-08-26,1.0,TV-14,Documentary,The 10-part documentary series from Steve Jame...,60.0,NaN,96.0,9,Universal acclaim,59,75,Mixed or average,NaN,"Participant,Kartemquin Films,Starz,Starz,Nolo ...","Bing Liu,Kevin Shaw,Steve James,Rebecca Parrish",NaN,"Kendale McCoy,Charles Donalson Jr.,Ke'Shawn Ku..."
2,1000357720,Planet Earth II,2016-11-06,1.0,TV-G,Documentary,"Narrated by David Attenborough, the sequel to ...",50.0,A new world revealed,96.0,10,Universal acclaim,92,242,Universal acclaim,NaN,"BBC Natural History Unit (NHU),BBC America,Zwe...","Justin Anderson,Ed Charles,Elizabeth White,Emm...",Elizabeth White,"David Attenborough,Chadden Hunter,Gordon Bucha..."
3,1000302375,The Staircase,2004-06,1.0,TV-MA,"Documentary,Crime,Drama",An 8-part documentary series about the celebra...,NaN,Did He Do It?,95.0,9,Universal acclaim,71,62,Generally favorable,NaN,"Maha Productions,ABC News,Docurama,Netflix,Net...",Jean-Xavier de Lestrade,"Jean-Xavier de Lestrade,Nathalie Sobania","Michael Peterson,David Rudolf,Ron Guerette,Mar..."
4,1000366530,The U.S. and the Holocaust,2022-09-18,1.0,TV-14,"Documentary,History","Narrated by Peter Coyote, the three-part docum...",133.0,NaN,96.0,10,Universal acclaim,53,65,Mixed or average,NaN,"Florentine Films,Public Broadcasting Service (...","Sarah Botstein,Ken Burns,Lynn Novick",Geoffrey C. Ward,"Peter Coyote,Daniel Mendelsohn,Peter Hayes,Deb..."


In [3]:
#drop any columns not being used
columns_to_keep = [
    'title', 'releaseDate', 'seasonCount', 'rating', 'genres',
    'metascore', 'metascore_count', 'userscore', 'userscore_count'
]

df_meta = data[columns_to_keep].copy()

df_meta.head()

,title,releaseDate,seasonCount,rating,genres,metascore,metascore_count,userscore,userscore_count
0,Planet Earth: Blue Planet II,2017-10-29,1.0,TV-G,Documentary,97.0,7,82,178
1,America to Me,2018-08-26,1.0,TV-14,Documentary,96.0,9,59,75
2,Planet Earth II,2016-11-06,1.0,TV-G,Documentary,96.0,10,92,242
3,The Staircase,2004-06,1.0,TV-MA,"Documentary,Crime,Drama",95.0,9,71,62
4,The U.S. and the Holocaust,2022-09-18,1.0,TV-14,"Documentary,History",96.0,10,53,65


In [5]:
#normalize titles accross both datasets, to be able to string match on join
import re
import unicodedata

def normalize_title(text):
    """
    Standardizes TV Show titles to create a reliable primary key between the 2 distinct datasets

    Due to Kaggle datasts having inconsistent formatting this pipeline is used to address issues such as:
      -variations in special characters: "Law & Order" vs. "Law and Order"
      -variations in accents: "Pokémon" vs "Pokemon"
      -any other punctuation and spacing discrepancies, that would make some joins fail
    """
    if pd.isna(text):
        return ""

    # Convert to lower case
    text = str(text).lower()

    # Standardize ampersands and convert hyphens/dashes to spaces
    text = text.replace('&', 'and')
    text = text.replace('-', ' ')

    # Strip accents and weird encoding artifacts (e.g., é becomes e)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # Strip ALL remaining punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Collapse multiple spaces into a single space, and trim edges
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Re-apply the function
df_meta['clean_title'] = df_meta['title'].apply(normalize_title)

# Check edge cases
df_meta.loc[[12, 16], ['title', 'clean_title']]

,title,clean_title
12,Homecoming: A Film by BeyoncÃ©,homecoming a film by beyonca
16,My So-Called Life,my so called life


In [6]:
df_meta[['title', 'clean_title']].head(10)

,title,clean_title
0,Planet Earth: Blue Planet II,planet earth blue planet ii
1,America to Me,america to me
2,Planet Earth II,planet earth ii
3,The Staircase,the staircase
4,The U.S. and the Holocaust,the us and the holocaust
5,Homicide: Life on the Street,homicide life on the street
6,The Office (UK),the office uk
7,The Larry Sanders Show,the larry sanders show
8,O.J.: Made in America,oj made in america
9,The Sopranos,the sopranos


In [7]:
# Load the Emmy Data
emmy_data = pd.read_csv('the_emmy_awards.csv', encoding='utf-8')

# Keep only columns we want to use
columns_to_keep_emmy = ['year', 'category', 'nominee', 'win']
df_emmy = emmy_data[columns_to_keep_emmy].copy()

# Apply same cleaning function to the 'nominee' column (lists title)
df_emmy['clean_title'] = df_emmy['nominee'].apply(normalize_title)

# Preview new title
df_emmy[['nominee', 'clean_title']].head()

,nominee,clean_title
0,The Simpsons,the simpsons
1,Family Guy,family guy
2,When You Wish Upon A Pickle: A Sesame Street S...,when you wish upon a pickle a sesame street sp...
3,F Is For Family,f is for family
4,Escape At Dannemora,escape at dannemora


In [8]:
# Perform an INNER JOIN on titles
matched_shows = pd.merge(df_meta, df_emmy, on='clean_title', how='inner')

# Count how many unique shows successfully matched
unique_matches = matched_shows['clean_title'].nunique()

print(f"Total unique shows matched across both datasets: {unique_matches}")

Total unique shows matched across both datasets: 687


In [9]:
## Create the unique list of shows
master_shows = matched_shows[['title', 'clean_title', 'releaseDate', 'rating', 'genres', 'metascore', 'metascore_count', 'userscore', 'userscore_count']].drop_duplicates(subset=['clean_title'])

# Assign a key to all shows
master_shows['show_id'] = range(1, len(master_shows) + 1)

#standardize to 3NF
# Table 1: The 'Shows' table
df_shows_table = master_shows[['show_id', 'title', 'releaseDate', 'rating', 'genres', 'metascore', 'metascore_count', 'userscore', 'userscore_count']]

# Table 2: The 'Awards' table
df_awards_table = pd.merge(df_emmy, master_shows[['clean_title', 'show_id']], on='clean_title', how='inner')
df_awards_table = df_awards_table[['show_id', 'category', 'win', 'year']]

In [10]:
# Drop rows where a single show has been nominated multiple times in one year
# for the same category. (Keeps the winning nominee if they won)

# Sort the dataframe by the 'win' column in descending order, forces all the True wins to the top
# of any cluster of identical show_id/category/year rows.
df_awards_table = df_awards_table.sort_values(by='win', ascending=False)

# Drop the duplicates based on composite primary key, keeping the 'first' row.
# if a show won the award, that True row is the one that is kept.
df_awards_table = df_awards_table.drop_duplicates(
    subset=['show_id', 'category', 'year'],
    keep='first'
)

# Optional: Sort by show_id and year just to keep the final CSV organized
df_awards_table = df_awards_table.sort_values(by=['show_id', 'year']).reset_index(drop=True)

In [11]:
# Export the finalized, 3NF compliant tables to CSV
df_shows_table.to_csv('shows.csv', index=False)
df_awards_table.to_csv('awards.csv', index=False)

print("Export complete. Ready for PostgreSQL.")

Export complete. Ready for PostgreSQL.
